# V83 - MLP on frozen E5

Pre-registered plan: the model matrix has one empty cell, nonlinear heads on dense embeddings. Linear probes of E5 are dead and nonlinear probes of sparse features are dead, but no version ran a nonlinear head on dense vectors. MLPRegressor per module on frozen E5 under grouped folds with a fresh seed, gated against word-only Ridge with the usual locks.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.neural_network import MLPRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
import re

SEED = 2222
N_SPLITS = 5
GATE_FLOOR = 0.002
rng = np.random.RandomState(SEED)

CANDIDATES = [Path("/kaggle/input/intelligo-data/data"), Path("data")]
DATA = next((p for p in CANDIDATES if (p / "train_relevance.csv").exists()), None)
assert DATA is not None, "train_relevance.csv not found"
OUT = Path("/kaggle/working" if Path("/kaggle/working").exists() else "results/v83")
OUT.mkdir(parents=True, exist_ok=True)
print("DATA:", DATA.resolve())

## Nonlinear dense heads vs linear sparse, same folds

E5 loads cached. Small MLP, fixed iterations, no tuning.

In [ ]:
MODS = [f"M_{i:03d}" for i in range(1, 18)]
train = pd.read_csv(DATA / "train_relevance.csv")
chats = pd.read_csv(DATA / "chat_history.csv")
assert len(train) == 4000
Y = train[MODS].to_numpy(float)
n_train = len(train)
by_user = chats.groupby("user_id")["user_chat_text"].apply(list)
docs = [" ".join(str(t) for t in by_user.get(u, [])) for u in list(train["user_id"])]
W = TfidfVectorizer(max_features=5000, ngram_range=(1, 1), min_df=3).fit_transform(docs).toarray()
E = None
for p in [Path("results/v58/e5_users.npy"), Path("/kaggle/input/intelligo-e5/e5_users.npy")]:
    if p.exists():
        E = np.load(p)[:n_train]
        print("E5 cache:", p)
        break
assert E is not None, "E5 cache required"

def template_group(texts):
    if not texts:
        return "NO_CHAT"
    return re.sub(r"\d+", "#", " ".join(str(texts[0]).lower().split()[:6]))
fold_groups = np.array([template_group(by_user.get(u, [])) for u in train["user_id"]])
strata = Y.argmax(1)

def ndcg5(yt, ys):
    order = np.argsort(-ys, axis=1)
    rel = np.take_along_axis(yt, order, 1)[:, :5]
    disc = 1.0 / np.log2(np.arange(2, 7))
    dcg = (rel * disc).sum(1)
    ideal = np.sort(yt, axis=1)[:, ::-1][:, :5]
    idcg = (ideal * disc).sum(1)
    return np.where(idcg > 0, dcg / np.maximum(idcg, 1e-12), 0.0)

oof_w, oof_m, fids = np.zeros_like(Y), np.zeros_like(Y), np.zeros(n_train, int)
skf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
for fi, (tr, va) in enumerate(skf.split(W, strata, fold_groups)):
    for j in range(len(MODS)):
        r = Ridge(alpha=1.0)
        r.fit(W[tr], Y[tr, j])
        oof_w[va, j] = r.predict(W[va])
        m = MLPRegressor(hidden_layer_sizes=(64,), max_iter=20, random_state=SEED)
        m.fit(E[tr], Y[tr, j])
        oof_m[va, j] = m.predict(E[va])
    fids[va] = fi + 1
    print(f"fold {fi + 1}: ridge={ndcg5(Y[va], oof_w[va]).mean():.5f} mlp-e5={ndcg5(Y[va], oof_m[va]).mean():.5f}", flush=True)
base = ndcg5(Y, oof_w)
d = ndcg5(Y, oof_m) - base
lo, hi = np.quantile([rng.choice(d, len(d), replace=True).mean() for _ in range(2000)], [0.025, 0.975])
wf = min(ndcg5(Y[fids == k], oof_m[fids == k]).mean() - base[fids == k].mean() for k in range(1, N_SPLITS + 1))
print(f"mlp-e5 OOF={ndcg5(Y, oof_m).mean():.5f} delta={d.mean():+.5f} CI95=[{lo:+.5f},{hi:+.5f}] worst-fold={wf:+.5f}")
print("ADOPT nonlinear dense" if (lo > GATE_FLOOR and wf >= 0) else "KEEP linear sparse; dense nonlinearity adds nothing")